In [2]:
import os
import shutil
import json

concatenated_json = "../concatenated_json"
output_root = "../addmypet_by_class"

os.makedirs(output_root, exist_ok=True)

copied = 0
skipped = 0

for filename in os.listdir(concatenated_json):
    if not filename.endswith(".json"):
        continue

    src = os.path.join(concatenated_json, filename)

    try:
        with open(src, "r") as f:
            data = json.load(f)
        cls = data[0].get("class") if isinstance(data, list) else data.get("class")
    except Exception as e:
        print(f"Could not read {filename}: {e}")
        skipped += 1
        continue

    if not cls:
        print(f"No 'class' field in {filename}, skipping.")
        skipped += 1
        continue

    out_dir = os.path.join(output_root, cls)
    os.makedirs(out_dir, exist_ok=True)

    shutil.copy2(src, os.path.join(out_dir, filename))
    copied += 1

classes_created = [d for d in os.listdir(output_root) if os.path.isdir(os.path.join(output_root, d))]
print(f"Created {len(classes_created)} class folders in '{output_root}': {sorted(classes_created)}")
print(f"Files copied: {copied}")
if skipped:
    print(f"Files skipped: {skipped}")

Created 54 class folders in '../addmypet_by_class': ['Acoelomorpha', 'Actinopterygii', 'Amphibia', 'Anopla', 'Anthozoa', 'Appendicularia', 'Arachnida', 'Archiacanthocephala', 'Ascidiacea', 'Asteroidea', 'Aves', 'Bivalvia', 'Branchiopoda', 'Cephalaspidomorphi', 'Cephalopoda', 'Chondrichthyes', 'Chromadorea', 'Clitellata', 'Copepoda', 'Cubozoa', 'Demospongiae', 'Echinoidea', 'Entognatha', 'Eutardigrada', 'Gastropoda', 'Gastrotricha', 'Gymnolaemata', 'Hexanauplia', 'Holothuroidea', 'Hydrozoa', 'Ichthyostraca', 'Insecta', 'Leptocardii', 'Malacostraca', 'Mammalia', 'Monogenea', 'Monogononta', 'Myxini', 'Nuda', 'Ophiuroidea', 'Ostracoda', 'Phoronida', 'Polychaeta', 'Polyplacophora', 'Pycnogonida', 'Reptilia', 'Rhynchonellata', 'Sagittoidea', 'Sarcopterygii', 'Scyphozoa', 'Secernentea', 'Tentaculata', 'Thaliacea', 'Turbellaria']
Files copied: 7332


In [2]:
import os

output_root = "../addmypet_by_class"

class_counts = {}
for cls in sorted(os.listdir(output_root)):
    cls_path = os.path.join(output_root, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.endswith(".json")])
        class_counts[cls] = count

for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"{cls:<35} {count} species")

print(f"\nTotal: {sum(class_counts.values())} species across {len(class_counts)} classes")

Actinopterygii                      2790 species
Aves                                1744 species
Mammalia                            1206 species
Reptilia                            370 species
Amphibia                            246 species
Chondrichthyes                      240 species
Bivalvia                            142 species
Branchiopoda                        107 species
Cephalopoda                         95 species
Gastropoda                          92 species
Malacostraca                        89 species
Insecta                             41 species
Hexanauplia                         15 species
Cephalaspidomorphi                  14 species
Echinoidea                          14 species
Clitellata                          12 species
Asteroidea                          11 species
Ascidiacea                          9 species
Polychaeta                          8 species
Arachnida                           7 species
Thaliacea                           7 species
Entogn

In [ ]:
import os
import re
import shutil

classes_folders    = "../addmypet_by_class"
pars_init_folder   = "../pars_init"
output_root        = "../../class_by_model"

# --- build species -> model lookup from pars_init files ---
model_pattern = re.compile(r"metaPar\.model\s*=\s*'(\w+)'")

species_model = {}
for fname in os.listdir(pars_init_folder):
    if not fname.endswith(".m"):
        continue
    # filename: pars_init_Genus_species.m  ->  Genus_species.json
    species_name = fname.replace("pars_init_", "").replace(".m", "")
    fpath = os.path.join(pars_init_folder, fname)
    with open(fpath, "r", errors="ignore") as f:
        content = f.read()
    match = model_pattern.search(content)
    if match:
        species_model[species_name] = match.group(1)

print(f"Model lookup built: {len(species_model)} species")

# --- organise by class then model ---
os.makedirs(output_root, exist_ok=True)

classes = [c for c in os.listdir(classes_folders) if os.path.isdir(os.path.join(classes_folders, c))]

copied  = 0
missing_model   = []
missing_json    = []

for cls in sorted(classes):
    cls_species_dir = os.path.join(classes_folders, cls)

    for species_file in os.listdir(cls_species_dir):
        species_name = species_file.replace(".json", "")
        model = species_model.get(species_name)

        if model is None:
            missing_model.append(species_name)
            continue

        src = os.path.join("../concatenated_json", species_file)
        if not os.path.isfile(src):
            missing_json.append(species_file)
            continue

        dest_dir = os.path.join(output_root, cls, model)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(src, os.path.join(dest_dir, species_file))
        copied += 1

# --- summary ---
print(f"\nFolder created : {output_root}")
print(f"Files copied   : {copied}")
print(f"Missing model  : {len(missing_model)}")
print(f"Missing JSON   : {len(missing_json)}")

print("\n--- Species per class per model ---")
for cls in sorted(os.listdir(output_root)):
    cls_path = os.path.join(output_root, cls)
    if not os.path.isdir(cls_path):
        continue
    models = sorted(os.listdir(cls_path))
    counts = {m: len(os.listdir(os.path.join(cls_path, m))) for m in models}
    counts_str = "  |  ".join(f"{m}: {n}" for m, n in counts.items())
    print(f"{cls:<35} {counts_str}")


In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

class_by_model = "../../class_by_model"
output_md  = "../../class_by_model/species_per_class_per_model.md"
output_png = "../../class_by_model/class_per_model.png"
output_svg = "../../class_by_model/class_per_model.svg"

# --- collect data ---
all_models = set()
class_counts = {}

for cls in sorted(os.listdir(class_by_model)):
    cls_path = os.path.join(class_by_model, cls)
    if not os.path.isdir(cls_path):
        continue
    counts = {}
    for m in os.listdir(cls_path):
        m_path = os.path.join(cls_path, m)
        if os.path.isdir(m_path):
            counts[m] = len(os.listdir(m_path))
            all_models.add(m)
    class_counts[cls] = counts

all_models = sorted(all_models)
classes    = sorted(class_counts.keys())

# --- markdown ---
header    = "| Class | " + " | ".join(all_models) + " | Total |"
separator = "|-------|" + "|".join(["-------"] * len(all_models)) + "|-------|"

rows = []
model_totals = {m: 0 for m in all_models}

for cls in classes:
    counts = class_counts[cls]
    total  = sum(counts.values())
    cells  = []
    for m in all_models:
        n = counts.get(m, 0)
        model_totals[m] += n
        cells.append(str(n) if n > 0 else "-")
    rows.append(f"| {cls} | " + " | ".join(cells) + f" | **{total}** |")

grand_total = sum(model_totals.values())
totals_row  = "| **Total** | " + " | ".join(str(model_totals[m]) for m in all_models) + f" | **{grand_total}** |"

with open(output_md, "w") as f:
    f.write("# Species per Class per Model\n\n")
    f.write(header + "\n")
    f.write(separator + "\n")
    f.write("\n".join(rows) + "\n")
    f.write(totals_row + "\n")

print(f"Saved {output_md}")

# --- heatmap ---
data = np.array([[class_counts[cls].get(m, 0) for m in all_models] for cls in classes], dtype=float)

# log scale for color (avoids large values drowning small ones)
data_log = np.where(data > 0, np.log1p(data), np.nan)

fig, ax = plt.subplots(figsize=(len(all_models) * 1.1 + 3, len(classes) * 0.45 + 2))

cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("whitesmoke")   # empty cells

im = ax.imshow(data_log, cmap=cmap, aspect="auto")

# annotate each cell
for i, cls in enumerate(classes):
    for j, m in enumerate(all_models):
        v = int(data[i, j])
        if v > 0:
            # pick text color for contrast
            norm_val = data_log[i, j] / np.nanmax(data_log)
            txt_color = "white" if norm_val > 0.6 else "black"
            ax.text(j, i, str(v), ha="center", va="center", fontsize=7.5, color=txt_color)

ax.set_xticks(range(len(all_models)))
ax.set_xticklabels(all_models, fontsize=9, fontweight="bold")
ax.set_yticks(range(len(classes)))
ax.set_yticklabels(classes, fontsize=8)
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

# colorbar
cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label("Species count (log scale)", fontsize=8)

ax.set_title("Species per Class per Model", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()

fig.savefig(output_png, dpi=150, bbox_inches="tight")
fig.savefig(output_svg, bbox_inches="tight")
plt.close()

print(f"Saved {output_png}")
print(f"Saved {output_svg}")


In [ ]:
import os
import shutil

class_by_model = "../../class_by_model"
output_root    = "../../class_by_model_filtered"

os.makedirs(output_root, exist_ok=True)

kept = 0
dropped = 0

for cls in sorted(os.listdir(class_by_model)):
    cls_path = os.path.join(class_by_model, cls)
    if not os.path.isdir(cls_path):
        continue

    # count species per model for this class
    model_counts = {}
    for m in os.listdir(cls_path):
        m_path = os.path.join(cls_path, m)
        if os.path.isdir(m_path):
            model_counts[m] = len(os.listdir(m_path))

    total = sum(model_counts.values())

    for m, n in model_counts.items():
        pct = 100 * n / total
        if pct > 10:
            src = os.path.join(cls_path, m)
            dst = os.path.join(output_root, cls, m)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            kept += 1
            print(f"  KEEP   {cls}/{m}  ({n}/{total} = {pct:.1f}%)")
        else:
            dropped += 1
            print(f"  DROP   {cls}/{m}  ({n}/{total} = {pct:.1f}%)")

print(f"\nDone. Models kept: {kept}  |  dropped: {dropped}")
print(f"Output: {output_root}")
